In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

from scripts.model.doNET import GATWithTransformerFusion, compute_graph_statistics_fast
from scripts.data_provider.graph_data_builder import build_pyg_data

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

## 1. Load new dataset
Gene names and protein names do **not** need to match the training data.
Different sequencing chemistries produce different gene/ADT panels — that is expected.
The model transfers via graph topology (k-NN structure) and a fixed-size HVG feature vector,
not via feature identities.

In [ ]:
import glob

base_path = "/projects/vanaja_lab/satya/Datasets/GSE220474"

rna_files   = glob.glob(os.path.join(base_path, "*_RNA_counts.csv.gz"))
sample_tags = sorted(set(f.split('_')[1] for f in [os.path.basename(x) for x in rna_files]))

rna_list, adt_list = [], []
for tag in sample_tags:
    print(f"Loading {tag}...")
    rna_path    = glob.glob(os.path.join(base_path, f"*_{tag}_RNA_counts.csv.gz"))[0]
    adt_path    = glob.glob(os.path.join(base_path, f"*_{tag}_ADT_counts.csv.gz"))[0]
    assign_path = glob.glob(os.path.join(base_path, f"*_{tag}_cell_sample_assign.csv.gz"))[0]

    meta_df = pd.read_csv(assign_path, index_col=0)

    rna_df = pd.read_csv(rna_path, index_col=0).T
    adt_df = pd.read_csv(adt_path, index_col=0).T

    adata_rna = ad.AnnData(rna_df)
    adata_rna.obs = meta_df.reindex(adata_rna.obs_names)
    adata_rna.obs['batch'] = tag
    adata_rna.var_names_make_unique()

    adata_adt = ad.AnnData(adt_df)
    adata_adt.obs = meta_df.reindex(adata_adt.obs_names)
    adata_adt.obs['batch'] = tag
    adata_adt.var_names_make_unique()

    rna_list.append(adata_rna)
    adt_list.append(adata_adt)

final_rna = ad.concat(rna_list, label='sample_batch', keys=sample_tags, join='inner')
final_adt = ad.concat(adt_list, label='sample_batch', keys=sample_tags, join='inner')
final_rna.obs_names_make_unique()
final_adt.obs_names_make_unique()

print(f"RNA: {final_rna.n_obs:,} cells x {final_rna.n_vars:,} genes")
print(f"ADT: {final_adt.n_obs:,} cells x {final_adt.n_vars:,} proteins")

## 2. Load trained model
Architecture is inferred directly from checkpoint weight shapes — no manual config.

In [ ]:
checkpoint = torch.load("deepomapnet_model.pt", map_location=device)
sd = checkpoint["model_state_dict"]

# Infer architecture from weight shapes
in_channels    = sd["gat_rna1.lin.weight"].shape[1]
out_channels   = sd["final_proj.4.weight"].shape[0]
# k.split('.')[2] is the layer index; [3] would be 'weight'/'bias' — count unique indices
num_layers     = len({k.split('.')[2] for k in sd
                      if k.startswith("transformer_fusion.rna_norms.")})
has_celltypes  = any(k.startswith("celltype_head") for k in sd)
num_cell_types = sd["celltype_head.4.weight"].shape[0] if has_celltypes else None

print(f"in_channels={in_channels}, out_channels={out_channels}, "
      f"num_layers={num_layers}, num_cell_types={num_cell_types}")

trained_model = GATWithTransformerFusion(
    in_channels=in_channels,
    hidden_channels=64,
    out_channels=out_channels,
    heads=4,
    dropout=0.4,
    nhead=4,
    num_layers=num_layers,
    use_adapters=True,
    use_positional_encoding=True,
    use_sparse_attention=True,
    num_cell_types=num_cell_types,
).to(device)

trained_model.load_state_dict(sd)
trained_model.eval()
print("Model loaded")

## 3. Preprocess RNA

Gene names vary across sequencing chemistries — we do **not** align by name.
We select the top `in_channels` highly variable genes from this dataset,
giving the model the same input dimensionality it was trained on.
The graph topology carries the transferable biological signal.

In [ ]:
rna = final_rna.copy()

# Standard normalisation before HVG selection
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

# Select top in_channels HVGs — same count as training, different genes
sc.pp.highly_variable_genes(rna, n_top_genes=in_channels, flavor='seurat_v3', span=1.0)
rna_hvg = rna[:, rna.var['highly_variable']].copy()
print(f"HVG subset: {rna_hvg.n_obs:,} cells x {rna_hvg.n_vars:,} genes")

# Z-score per gene (in-place)
X = rna_hvg.X.toarray() if hasattr(rna_hvg.X, 'toarray') else rna_hvg.X.copy()
X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)
rna_hvg.X = X
print("RNA z-scored")

## 4. Build graphs

**RNA graph**: k-NN built in HVG space; raw HVG features become node features for `gat_rna1`.

**ADT graph**: k-NN built from new ADT counts — protein names are irrelevant because
`gat_adt_init` derives its node features from RNA embeddings, not ADT expression
(see `doNET.py:642`). Only the edge topology transfers.

In [ ]:
# RNA — raw HVG features, no PCA (model expects in_channels-dim input)
rna_pyg = build_pyg_data(rna_hvg, use_pca=False)
print(f"RNA graph — nodes: {rna_pyg.num_nodes:,}  edges: {rna_pyg.num_edges:,}  "
      f"features: {rna_pyg.num_node_features}")

# ADT — topology only; protein names and counts don't need to match training
adt = final_adt.copy()
adt.X = adt.X.astype(float)
sc.pp.normalize_total(adt, target_sum=1e4)
sc.pp.log1p(adt)
adt_pyg = build_pyg_data(adt, use_pca=True)
print(f"ADT graph — nodes: {adt_pyg.num_nodes:,}  edges: {adt_pyg.num_edges:,}")

assert rna_pyg.num_nodes == adt_pyg.num_nodes, \
    f"Cell count mismatch: RNA={rna_pyg.num_nodes}, ADT={adt_pyg.num_nodes}"

# Graph statistics for positional encoding
num_nodes = rna_pyg.num_nodes
node_degrees_rna, clustering_coeffs_rna = compute_graph_statistics_fast(
    rna_pyg.edge_index, num_nodes)
node_degrees_adt, clustering_coeffs_adt = compute_graph_statistics_fast(
    adt_pyg.edge_index, num_nodes)
print("Graph statistics computed")

## 5. Run inference

In [ ]:
rna_pyg = rna_pyg.to(device)
adt_pyg = adt_pyg.to(device)

trained_model.eval()
with torch.no_grad():
    adt_pred, aml_pred, fused_emb = trained_model(
        x=rna_pyg.x,
        edge_index_rna=rna_pyg.edge_index,
        edge_index_adt=adt_pyg.edge_index,
        node_degrees_rna=node_degrees_rna.to(device),
        node_degrees_adt=node_degrees_adt.to(device),
        clustering_coeffs_rna=clustering_coeffs_rna.to(device),
        clustering_coeffs_adt=clustering_coeffs_adt.to(device),
    )

# Denormalize using training-time CLR stats stored in checkpoint
adt_mean_ckpt = checkpoint["normalization"]["adt_mean"].to(device)
adt_std_ckpt  = checkpoint["normalization"]["adt_std"].to(device)
adt_pred_np   = (adt_pred * adt_std_ckpt + adt_mean_ckpt).cpu().numpy()

aml_probs  = torch.sigmoid(aml_pred).cpu().numpy().flatten()
aml_labels = (aml_probs > 0.5).astype(int)

print(f"Inference complete")
print(f"ADT predictions : {adt_pred_np.shape}")
print(f"AML positive    : {aml_labels.sum():,} / {num_nodes:,} cells")

## 6. Save results

In [ ]:
# Use real protein names from checkpoint; validate shape before use
_adt_names = checkpoint.get("adt_names")
n_pred = adt_pred_np.shape[1]

if _adt_names and len(_adt_names) == n_pred:
    protein_cols = _adt_names
else:
    if _adt_names:
        print(f"WARNING: checkpoint has {len(_adt_names)} protein names but model "
              f"outputs {n_pred} dimensions — checkpoint adt_names is stale. "
              f"Re-save the checkpoint from Training.ipynb cell 20 and re-run.")
    protein_cols = [f"protein_{i}" for i in range(n_pred)]

results = pd.DataFrame(
    adt_pred_np,
    index=final_rna.obs_names,
    columns=protein_cols,
)
results["aml_prob"]  = aml_probs
results["aml_label"] = aml_labels
results.to_csv("predictions.csv")

# AnnData with fused embeddings for downstream UMAP / clustering
out_adata = final_rna.copy()
out_adata.obsm["X_deepomapnet"] = fused_emb.cpu().numpy()
out_adata.obs["aml_prob"]       = aml_probs
out_adata.obs["aml_label"]      = aml_labels.astype(str)
out_adata.write_h5ad("predictions.h5ad")

print("Saved predictions.csv and predictions.h5ad")
print(f"Protein columns (first 5): {protein_cols[:5]}")
print(results.head())

## 7. UMAP visualisation

In [ ]:
import umap

print("Fitting UMAP on fused embeddings...")
reducer     = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
umap_coords = reducer.fit_transform(fused_emb.cpu().numpy())
print(f"UMAP done — {umap_coords.shape[0]:,} cells")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc1 = axes[0].scatter(umap_coords[:, 0], umap_coords[:, 1],
                      c=aml_probs, cmap='RdYlBu_r', s=1.5, rasterized=True, linewidths=0)
plt.colorbar(sc1, ax=axes[0], label='AML probability')
axes[0].set_title('AML probability')
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].scatter(umap_coords[:, 0], umap_coords[:, 1],
                c=aml_labels, cmap='bwr', s=1.5, rasterized=True, linewidths=0)
axes[1].set_title('AML prediction (binary)')
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout()
plt.savefig("umap_predictions.pdf", bbox_inches='tight', dpi=150)
plt.show()
print("Saved umap_predictions.pdf")

## 8. Predicted ADT Marker Visualizations

In [ ]:
adt_names = checkpoint.get("adt_names") or [f"protein_{i}" for i in range(adt_pred_np.shape[1])]
n_proteins = adt_pred_np.shape[1]
print(f"Proteins to visualize: {n_proteins}")
print(f"Protein names (first 5): {adt_names[:5]}")

In [ ]:
# --- UMAP colored by top predicted proteins ---
# Select proteins with highest variance across cells (most informative to show)
variances = adt_pred_np.var(axis=0)
top_k = min(9, n_proteins)
top_idx = variances.argsort()[::-1][:top_k]

ncols = 3
nrows = int(np.ceil(top_k / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
axes = np.array(axes).flatten()

for rank, pidx in enumerate(top_idx):
    ax = axes[rank]
    vals = adt_pred_np[:, pidx]
    sc = ax.scatter(
        umap_coords[:, 0], umap_coords[:, 1],
        c=vals, cmap="RdYlBu_r", s=2, rasterized=True, linewidths=0,
    )
    plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
    ax.set_title(adt_names[pidx], fontsize=10, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("UMAP 1", fontsize=8); ax.set_ylabel("UMAP 2", fontsize=8)

for j in range(rank + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Predicted ADT Marker Expression on UMAP\n(top proteins by variance)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.savefig("umap_adt_markers.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved umap_adt_markers.pdf")

In [ ]:
# --- Violin plots: predicted marker levels split by AML status ---
top_violin = min(12, n_proteins)
top_violin_idx = variances.argsort()[::-1][:top_violin]

ncols = 4
nrows = int(np.ceil(top_violin / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.2))
axes = np.array(axes).flatten()

normal_mask = aml_labels == 0
aml_mask    = aml_labels == 1

for rank, pidx in enumerate(top_violin_idx):
    ax = axes[rank]
    vals_normal = adt_pred_np[normal_mask, pidx]
    vals_aml    = adt_pred_np[aml_mask,    pidx]

    parts = ax.violinplot(
        [vals_normal, vals_aml],
        positions=[0, 1],
        showmedians=True,
        showextrema=False,
    )
    for pc, color in zip(parts["bodies"], ["#2196F3", "#F44336"]):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts["cmedians"].set_color("black")
    parts["cmedians"].set_linewidth(1.5)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Normal", "AML"], fontsize=8)
    ax.set_title(adt_names[pidx], fontsize=9, fontweight="bold")
    ax.set_ylabel("Predicted expression", fontsize=7)
    ax.grid(axis="y", alpha=0.3)

for j in range(rank + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Predicted ADT Expression by AML Status\n(top proteins by variance)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.savefig("violin_adt_aml.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved violin_adt_aml.pdf")

In [ ]:
# --- Heatmap: mean predicted protein expression (AML vs Normal) ---
show_k = min(25, n_proteins)
top_heatmap_idx = variances.argsort()[::-1][:show_k]

mean_normal = adt_pred_np[normal_mask][:, top_heatmap_idx].mean(axis=0)
mean_aml    = adt_pred_np[aml_mask][:, top_heatmap_idx].mean(axis=0)
heatmap_data = np.stack([mean_normal, mean_aml], axis=0)  # [2, P]

# Row-normalise each protein to [0, 1] for visual comparison
col_min  = heatmap_data.min(axis=0, keepdims=True)
col_max  = heatmap_data.max(axis=0, keepdims=True)
heatmap_norm = (heatmap_data - col_min) / (col_max - col_min + 1e-8)

heatmap_names = [adt_names[i] for i in top_heatmap_idx]

fig, ax = plt.subplots(figsize=(max(10, show_k * 0.55), 3.5))
im = ax.imshow(heatmap_norm, cmap="RdYlBu_r", aspect="auto", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8, label="Normalised mean expression")

ax.set_yticks([0, 1])
ax.set_yticklabels(["Normal", "AML"], fontsize=11)
ax.set_xticks(range(show_k))
ax.set_xticklabels(heatmap_names, rotation=45, ha="right", fontsize=8)
ax.set_title(f"Mean Predicted ADT Expression — Normal vs AML (top {show_k} proteins)",
             fontsize=12, fontweight="bold")

# Annotate cells with raw mean values
for row_i, row_label in enumerate(["Normal", "AML"]):
    for col_j, pidx in enumerate(top_heatmap_idx):
        val = heatmap_data[row_i, col_j]
        ax.text(col_j, row_i, f"{val:.2f}", ha="center", va="center",
                fontsize=6.5, color="black")

fig.tight_layout()
plt.savefig("heatmap_adt_aml.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved heatmap_adt_aml.pdf")

## 9. True vs Predicted Protein Expression

In [ ]:
import math
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score
from scripts.data_provider.data_preprocessing import clr_normalize

# ── 1. Prepare true ADT in CLR space (full panel) ────────────────────────────
adt_for_clr = final_adt.copy()
adt_for_clr.X = adt_for_clr.X.astype(float)
adt_true_clr = clr_normalize(adt_for_clr, axis=1)
adt_true_np  = (
    adt_true_clr.X.toarray()
    if hasattr(adt_true_clr.X, "toarray")
    else adt_true_clr.X.copy()
)
true_names = list(final_adt.var_names)
pred_names = protein_cols

# ── 2. Find matched proteins ──────────────────────────────────────────────────
common = sorted(set(true_names) & set(pred_names))
print(f"True panel  : {len(true_names)} proteins")
print(f"Pred panel  : {len(pred_names)} proteins")
print(f"Matched     : {len(common)} proteins")
print(f"Matched names: {common}")

if len(common) == 0:
    raise ValueError(
        "No protein names overlap. Check that adt_names in the checkpoint "
        "match the test dataset var_names."
    )

true_idx = [true_names.index(p) for p in common]
pred_idx = [pred_names.index(p) for p in common]

true_matched = adt_true_np[:, true_idx]   # (cells, n_common) — CLR-137 space
pred_matched = adt_pred_np[:, pred_idx]   # (cells, n_common) — CLR-279 space

# ── 3. Fix CLR reference — re-centre both over the matched subset ─────────────
# CLR_full_j = log1p(raw_j) - mean_full(log1p)
# Subtracting per-cell mean of the matched subset gives:
#   CLR_full_j - mean_matched(CLR_full) = log1p(raw_j) - mean_matched(log1p)
# which is exactly CLR computed over the matched panel — same reference for both.
true_matched = true_matched - true_matched.mean(axis=1, keepdims=True)
pred_matched = pred_matched - pred_matched.mean(axis=1, keepdims=True)
print("CLR reference corrected to matched-26 panel for both true and predicted.")

# ── 4. Per-protein metrics ────────────────────────────────────────────────────
records = []
for i, name in enumerate(common):
    tv, pv     = true_matched[:, i], pred_matched[:, i]
    r, _       = pearsonr(tv, pv)
    rho, _     = spearmanr(tv, pv)
    r2         = r2_score(tv, pv)
    records.append({"protein": name, "pearson_r": r, "spearman_rho": rho, "R2": r2})

import pandas as pd
metrics_df = pd.DataFrame(records).sort_values("pearson_r", ascending=False)
print(f"\n{'Metric':<14} {'Mean':>8} {'Median':>8} {'Min':>8} {'Max':>8}")
for col in ["pearson_r", "spearman_rho", "R2"]:
    v = metrics_df[col]
    print(f"{col:<14} {v.mean():>8.4f} {v.median():>8.4f} {v.min():>8.4f} {v.max():>8.4f}")
print()
print(metrics_df.to_string(index=False))

# ── 5. Visualisations ─────────────────────────────────────────────────────────
n_common = len(common)
ncols    = min(4, n_common)
nrows    = math.ceil(n_common / ncols)

# Panel A — per-protein scatter (true vs predicted, coloured by AML label)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
axes = np.array(axes).flatten() if n_common > 1 else [axes]

for i, name in enumerate(common):
    ax = axes[i]
    ax.scatter(true_matched[:, i], pred_matched[:, i],
               c=aml_labels, cmap="bwr", s=2, alpha=0.5, rasterized=True, linewidths=0)
    mn = min(true_matched[:, i].min(), pred_matched[:, i].min())
    mx = max(true_matched[:, i].max(), pred_matched[:, i].max())
    ax.plot([mn, mx], [mn, mx], "k--", lw=1)
    r  = metrics_df.loc[metrics_df.protein == name, "pearson_r"].values[0]
    r2 = metrics_df.loc[metrics_df.protein == name, "R2"].values[0]
    ax.set_title(f"{name}\nr={r:.3f}  R²={r2:.3f}", fontsize=9, fontweight="bold")
    ax.set_xlabel("True CLR-26", fontsize=8)
    ax.set_ylabel("Predicted CLR-26", fontsize=8)
    ax.grid(alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("True vs Predicted ADT — Matched Proteins (CLR-26 reference)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("scatter_matched_proteins.pdf", bbox_inches="tight", dpi=150)
plt.show()

# Panel B — Pearson r bar chart
fig, ax = plt.subplots(figsize=(max(6, n_common * 0.7), 4))
colors = ["#2196F3" if r >= 0 else "#F44336" for r in metrics_df["pearson_r"]]
ax.bar(metrics_df["protein"], metrics_df["pearson_r"], color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Pearson r", fontsize=11)
ax.set_title(f"Per-protein Pearson r — {n_common} matched proteins (CLR-26)\n"
             f"mean={metrics_df['pearson_r'].mean():.3f}", fontsize=11, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("barplot_matched_pearson.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Saved scatter_matched_proteins.pdf and barplot_matched_pearson.pdf")